# 04c — Post-LoRA Attribution

**Runs on RunPod GPU. Always start from a FRESH kernel.**

Loads the merged LoRA checkpoint (`my_work/checkpoints/lora_combined_merged/`)
produced by `04b_lora_training.ipynb` and runs attribution on the frozen
probe set (`prompts_triangle_v2.jsonl`).

**Why a separate notebook?**  Training leaves ~12 GB of GPU memory that cannot
be freed without a kernel restart (`empty_cache()` only releases the PyTorch
cache, not memory held by the CUDA process).  Starting fresh gives the full
19–24 GB to Gemma-2-2B + transcoders + attribution.

**Prerequisites:**
- `04_lora_training_data.ipynb` run (probe JSONL exists)
- `04b_lora_training.ipynb` run through §7 (`lora_combined_merged/` exists)

**Outputs:**
- Pilot: `my_work/results/statistics/stats_lora_combined_pilot.json`
- Full:  `my_work/results/statistics/stats_lora_combined.json`

## 0 — Environment setup

In [ ]:
import os
import sys
from pathlib import Path

# Must be set BEFORE torch is imported anywhere.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IN_RUNPOD = os.environ.get("RUNPOD_POD_ID") is not None

def _find_repo_root():
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "circuit_tracer" / "__init__.py").is_file():
            return directory
    workspace = Path("/workspace")
    if workspace.is_dir():
        for child in workspace.iterdir():
            if child.is_dir() and (child / "circuit_tracer" / "__init__.py").is_file():
                return child
    repo_override = os.environ.get("CT_REPO_DIR")
    if repo_override:
        p = Path(repo_override).expanduser().resolve()
        if (p / "circuit_tracer" / "__init__.py").is_file():
            return p
    return None

_root = _find_repo_root()
if _root is not None:
    if str(_root) not in sys.path:
        sys.path.insert(0, str(_root))
    _my_work = _root / "my_work"
    if str(_my_work) not in sys.path:
        sys.path.insert(0, str(_my_work))
    print(f"Repo root: {_root}")
else:
    print("WARNING: could not locate circuit_tracer repo. Set CT_REPO_DIR.")

try:
    from dotenv import load_dotenv
    if _root is not None and (_root / ".env").is_file():
        load_dotenv(_root / ".env")
except ImportError:
    pass

MY_WORK = _my_work if _root else Path(".").resolve()

if IN_RUNPOD:
    persistent_root = Path(os.environ.get("RUNPOD_PERSISTENT_ROOT", "/workspace")).resolve()
    hf_home = persistent_root / "hf"
    for key, path in {
        "HF_HOME": hf_home,
        "HUGGINGFACE_HUB_CACHE": hf_home / "hub",
        "TRANSFORMERS_CACHE": hf_home / "transformers",
    }.items():
        path.mkdir(parents=True, exist_ok=True)
        os.environ[key] = str(path)

STATS_DIR  = MY_WORK / "results" / "statistics"
GRAPHS_DIR = MY_WORK / "results" / "graphs_lora_combined"
STATS_DIR.mkdir(parents=True, exist_ok=True)
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

print(f"MY_WORK  : {MY_WORK}")

## 1 — Constants

In [ ]:
import json
import time
import gc
import traceback
import random

import torch

MODEL_NAME      = "google/gemma-2-2b"
TRANSCODER_NAME = "gemma"
TOKEN_TRUE      = " True"
TOKEN_FALSE     = " False"
VOCAB_ID_TRUE   = 5569
VOCAB_ID_FALSE  = 7662

ATTR_KWARGS = dict(
    batch_size=256,
    max_feature_nodes=8192,
    offload="disk",
    verbose=False,
)
PHASE = "lora_combined"

# Pilot vs full
# Start with RUN_FULL=False (~40 prompts). If pilot succeeds, set True and re-run from §4.
RUN_FULL = False
N_PILOT  = 40

MERGED_DIR       = MY_WORK / "checkpoints" / "lora_combined_merged"
PROBE_JSONL      = MY_WORK / "data" / "prompts_triangle_v2.jsonl"
PILOT_STATS_FILE = STATS_DIR / "stats_lora_combined_pilot.json"
FULL_STATS_FILE  = STATS_DIR / "stats_lora_combined.json"
STATS_FILE       = FULL_STATS_FILE if RUN_FULL else PILOT_STATS_FILE

assert MERGED_DIR.exists(), (
    f"Merged checkpoint not found: {MERGED_DIR}\n"
    "Run 04b_lora_training.ipynb through §7 first."
)

device_str = "cuda" if torch.cuda.is_available() else "cpu"
device     = torch.device(device_str)

print(f"MERGED_DIR : {MERGED_DIR}")
print(f"STATS_FILE : {STATS_FILE}")
print(f"RUN_FULL   : {RUN_FULL}  (pilot N={N_PILOT})")
print(f"CUDA       : {torch.cuda.is_available()}", end="")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"  |  Free VRAM: {free/1e9:.1f} GB / {total/1e9:.1f} GB")
else:
    print()

## 2 — Load merged model into ReplacementModel

TransformerLens requires a registered `model_name`; pass the local merged
weights via `hf_model=`.  We load `merged_hf` on CPU so that only one copy
of Gemma-2-2B lives on the GPU (inside HookedTransformer) at any time.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from circuit_tracer import ReplacementModel

print("Loading merged weights on CPU ...")
merged_hf = AutoModelForCausalLM.from_pretrained(
    str(MERGED_DIR),
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map="cpu",
).eval()

print("Building ReplacementModel (moves weights to GPU) ...")
ct_model = ReplacementModel.from_pretrained(
    MODEL_NAME,
    TRANSCODER_NAME,
    dtype=torch.bfloat16,
    backend="transformerlens",
    device=device,
    lazy_encoder=True,
    lazy_decoder=True,
    hf_model=merged_hf,
)

# Free the CPU copy immediately — HookedTransformer now holds the weights on GPU.
del merged_hf
gc.collect()

ct_tokenizer = ct_model.tokenizer

id_true  = ct_tokenizer.encode(TOKEN_TRUE,  add_special_tokens=False)[-1]
id_false = ct_tokenizer.encode(TOKEN_FALSE, add_special_tokens=False)[-1]
assert id_true  == VOCAB_ID_TRUE,  f"Token mismatch True:  {id_true}"
assert id_false == VOCAB_ID_FALSE, f"Token mismatch False: {id_false}"
print(f"ReplacementModel ready. Token IDs: True={id_true}, False={id_false}")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM after load: {(total-free)/1e9:.1f} GB used / {total/1e9:.1f} GB total")

## 3 — Load probe set and select pilot subset

In [ ]:
from collections import Counter, defaultdict

all_probe = []
with open(PROBE_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line.strip())
        if row.get("task_type", "binary") == "binary":
            all_probe.append(row)

print(f"Binary probe prompts: {len(all_probe)}")

if RUN_FULL:
    probe_prompts = all_probe
    print(f"RUN_FULL=True: running all {len(probe_prompts)} binary prompts.")
else:
    strata: dict = defaultdict(list)
    for p in all_probe:
        key = (p.get("family", "?"), p.get("tail", "?"), str(p.get("label", "?")))
        strata[key].append(p)

    rng = random.Random(42)
    pilot = []
    per_stratum = max(1, N_PILOT // len(strata))
    for key, bucket in sorted(strata.items()):
        rng.shuffle(bucket)
        pilot.extend(bucket[:per_stratum])

    remaining_pool = [p for p in all_probe if p not in pilot]
    rng.shuffle(remaining_pool)
    while len(pilot) < N_PILOT and remaining_pool:
        pilot.append(remaining_pool.pop(0))

    probe_prompts = pilot[:N_PILOT]
    dist = Counter(
        (p.get("family", "?"), p.get("tail", "?"), str(p.get("label", "?")))
        for p in probe_prompts
    )
    print(f"Pilot subset: {len(probe_prompts)} prompts")
    for key, cnt in sorted(dist.items()):
        print(f"  {key}: {cnt}")

## 4 — First-token accuracy (post-LoRA)

In [ ]:
from utils.graph_statistics import _binary_label_true

correct = 0
true_correct = 0; true_total = 0
false_correct = 0; false_total = 0
predictions = []

with torch.inference_mode():
    for entry in probe_prompts:
        logits, _ = ct_model.feature_intervention(entry["prompt"], [])
        pred_id = int(logits.squeeze(0)[-1].argmax().item())
        pred_tok = ct_tokenizer.decode([pred_id])
        expected_id = VOCAB_ID_TRUE if entry["label"] else VOCAB_ID_FALSE
        is_ok = (pred_id == expected_id)
        if is_ok:
            correct += 1
        if entry["label"]:
            true_total += 1
            if pred_id == VOCAB_ID_TRUE:
                true_correct += 1
        else:
            false_total += 1
            if pred_id == VOCAB_ID_FALSE:
                false_correct += 1
        predictions.append({
            "prompt_id": entry["prompt_id"],
            "label": entry["label"],
            "pred_token": pred_tok,
            "pred_id": pred_id,
            "is_correct": is_ok,
        })

n = len(probe_prompts)
print(f"Post-LoRA first-token accuracy (n={n}):")
print(f"  Overall : {correct}/{n} = {correct/n:.1%}")
if true_total:
    print(f"  True    : {true_correct}/{true_total} = {true_correct/true_total:.1%}")
if false_total:
    print(f"  False   : {false_correct}/{false_total} = {false_correct/false_total:.1%}")
print()
print("Baseline first-token accuracy was ~18.5% on binary prompts.")
print("An increase here confirms the LoRA changed the output distribution.")

## 5 — Attribution + stats loop

Identical to notebook `02`. Stats schema is identical — join on `prompt_id` in `05_lora_comparison.ipynb`.
Resume logic: prompts already in the stats file are skipped.

In [ ]:
import importlib
import utils.graph_statistics as gs_mod
importlib.reload(gs_mod)

from circuit_tracer import attribute
from utils.graph_statistics import compute_statistics, load_statistics, append_statistic

existing = load_statistics(STATS_FILE)
done_ids = {e["prompt_id"] for e in existing if e.get("attribution_succeeded")}
n_prev_failed = sum(1 for e in existing if not e.get("attribution_succeeded"))
remaining = [p for p in probe_prompts if p["prompt_id"] not in done_ids]

print(f"Stats file     : {STATS_FILE}")
print(f"Already done   : {len(done_ids)} succeeded")
print(f"Previously failed: {n_prev_failed} (will retry)")
print(f"Remaining      : {len(remaining)} prompts")
print()

t0 = time.time()
n_success = 0
n_fail = 0
progress_every = 5

for i, entry in enumerate(remaining, start=1):
    pid = entry["prompt_id"]
    print(f"[{i}/{len(remaining)}] {pid} ...", end=" ", flush=True)

    try:
        graph = attribute(
            prompt=entry["prompt"],
            model=ct_model,
            attribution_targets=[TOKEN_TRUE, TOKEN_FALSE],
            **ATTR_KWARGS,
        )
        stat = compute_statistics(graph, entry, phase=PHASE)
        append_statistic(stat, STATS_FILE)
        n_success += 1

        del graph
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        density  = stat.get("edge_density", float("nan"))
        n_active = stat.get("n_active_features", "?")
        print(f"OK  n_active={n_active}  density={density:.4f}")

    except Exception as exc:
        n_fail += 1
        stat = {
            "prompt_id": pid,
            "attribution_succeeded": False,
            "error": str(exc)[:200],
            "phase": PHASE,
        }
        append_statistic(stat, STATS_FILE)
        print(f"FAIL: {exc}")
        traceback.print_exc()

    if i % progress_every == 0:
        elapsed = time.time() - t0
        speed   = i / elapsed
        eta     = (len(remaining) - i) / speed if speed > 0 else float("inf")
        print(
            f"[progress] {i}/{len(remaining)} ({i/len(remaining):.0%}) | "
            f"success={n_success}/{n_success+n_fail} | "
            f"{speed:.3f} prompts/s | ETA {eta/60:.1f} min"
        )

print()
print(f"Done. {n_success} succeeded, {n_fail} failed in {time.time()-t0:.0f}s.")
print(f"Stats saved to: {STATS_FILE}")

## 6 — Scale-up decision

After the pilot: if `n_success >= N_PILOT * 0.9` and stats look sane,
set `RUN_FULL = True` in §1 and **re-run from §1** to run all 270 binary
prompts.  Resume logic skips already-completed IDs.

In [ ]:
final_stats = load_statistics(STATS_FILE)
n_ok    = sum(1 for s in final_stats if s.get("attribution_succeeded"))
n_total = len(final_stats)

print(f"=== Post-run summary ===")
print(f"Stats file : {STATS_FILE}")
print(f"Total rows : {n_total}")
print(f"Succeeded  : {n_ok}  ({n_ok/max(n_total,1):.1%})")
print()
if not RUN_FULL:
    print("Pilot complete.")
    print("If results look good: set RUN_FULL=True in §1 and re-run from §1 onwards.")
    print("Full run saves to:", FULL_STATS_FILE)
else:
    print("Full run complete. Proceed to 05_lora_comparison.ipynb.")